In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

db_path = Path("chroma_db")
chroma_client = chromadb.PersistentClient(path=str(db_path))

resumes_path = Path("resumes")
resume_files = [f for f in resumes_path.iterdir() if f.is_file()]

print(f"Client OpenAI pronto. Trovati {len(resume_files)} file nella cartella resumes:")
for f in resume_files:
    print(f" - {f.name}")
    
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

collection = chroma_client.get_or_create_collection(name="hr_resumes")

for f in resume_files:
    if f.suffix.lower() == ".txt":
        with open(f, "r", encoding="utf-8") as file:
            content = file.read()
            
            chunks = chunk_text(content)
            
            for idx, chunk in enumerate(chunks):
                chunk_id = f"{f.name}_chunk_{idx}"
                collection.add(
                    documents=[chunk],
                    ids=[chunk_id],
                    metadatas=[{"source": f.name}]
                )

print(f"Database vettoriale popolato! Totale elementi nella collection: {collection.count()}")

user_query = "Chi ha competenze di Python e lavora nel settore tech?"

results = collection.query(
    query_texts=[user_query],
    n_results=2
)

retrieved_chunks = results["documents"][0]
sources = results["metadatas"][0]

print("--- ASSISTENTE IN FUNZIONE ---")
for doc, src in zip(retrieved_chunks, sources):
    print(f"Fonte: {src['source']}")
    print(f"Testo: {doc}\n")

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Sei un assistente HR esperto. Rispondi alla domanda dell'utente basandoti unicamente sul contesto estratto dai curriculum forniti di seguito.

Contesto:
{context}

Domanda: {user_query}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Sei un assistente HR preciso e professionale."},
        {"role": "user", "content": prompt}
    ]
)

print("--- MATCH TROVATO ---")
print(response.choices[0].message.content)

Client OpenAI pronto. Trovati 8 file nella cartella resumes:
 - CV1.txt
 - CV2.txt
 - CV3.txt
 - CV4.txt
 - CV5.txt
 - CV6.txt
 - CV7.txt
 - CV8.txt
Database vettoriale popolato! Totale elementi nella collection: 17
--- ASSISTENTE IN FUNZIONE ---
Fonte: CV7.txt
Testo: ZA PROFESSIONALE
Lead Software Architect | TechScale S.p.A. | 2019 - Presente
- Guida di un team di 20 sviluppatori in progetti enterprise.
- Progettazione e implementazione di architettura microservizi che gestisce 5M+ utenti.
### COMPETENZE TECNICHE
- JavaScript/TypeScript
- Python
- Java
- React.js / Node.js

Fonte: CV6.txt
Testo: r.l. | 2018 - Presente
- Gestione di un portafoglio clienti chiave del valore di oltre 4M€ annui.
- Apertura di 15 nuovi mercati regionali con una crescita del fatturato del 40%.
### COMPETENZE TECNICHE
- B2B Sales & Negotiation
- CRM Management (Salesforce, HubSpot)
- Strategic Planning
- Lead Generation



ImportError: cannot import name 'solve_response_format_t' from 'openai.lib._parsing' (c:\Users\adamo\wa\Python\assistant\.venv\Lib\site-packages\openai\lib\_parsing\__init__.py)